# Turtle Breakout 20 — flat-day legs with turtle exits: long after a sharp drop (A), short at a stalled peak (B)

Pre-registration: `discussions/Turtle_Breakout/022_claude_tb20_preregistration.md`. Code: `turtle_flat_legs.py`.

Signals only at the close of a FLAT day (the TB 01 turtle holds no long). **A (long):** today's return ≤ 20th percentile of the previous 365 FLAT days. **B (short):** close within 1 ATR of the 55-day high **and** |20-day return| ≤ 0.5 × its usual size.
**Exits = the turtle's:** fixed stop 2 ATR, trailing 10-day low/high, **no take-profit, no time limit**. Size = turtle (2% of HWM / 2 ATR, ≤ 3×). 6 bp per side; funding on the perpetual.
**Gates per leg** (Yahoo 2014–19, Yahoo 2020–, Binance perp): mean net R above random FLAT-day entries with the same exits, p < 0.01, and mean net return > 0.

Environment: `TB17_DATA`, `TB01_DATA_DIR`, `TB20_SYNTHETIC=1`.

In [ ]:
import os, sys, json, subprocess, tempfile, warnings, platform
from dataclasses import replace
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from dotenv import load_dotenv
warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'data_engineering').is_dir(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src')); load_dotenv(ROOT / '.env')
from data_engineering.modeling import turtle_breakout as tb, turtle_reverse as tr, turtle_stocks as tk, turtle_stock_accounts as ta, turtle_btc_history as bh, turtle_flat_diag as fd, turtle_reversal1d as rv, turtle_flat_legs as fl

SYNTHETIC = os.getenv('TB20_SYNTHETIC', '0') == '1'
LP = fl.LParams(); BP = bh.BParams()
RV = replace(rv.RParams(), quantile=LP.a_quantile, lookback_flat_days=LP.a_lookback, min_flat_days=LP.a_min_flat)
P = replace(tb.Params(), min_bar_share=0.0, funding=False, fee=BP.side_cost - tb.Params().slip); RPt = replace(tr.RParams(), side_cost=BP.side_cost)
if SYNTHETIC:
    OUT = Path(tempfile.mkdtemp(prefix='tb20-')) / '20'; LP = replace(LP, n_null=2000)
    d, _ = tk.synthetic_daily(4400, seed=20, vol=0.04, gap=0.0, steps=48, drift=0.0008); d.index = pd.date_range('2014-09-17', periods=len(d), freq='D'); YH = d
    BN = YH[YH.index >= '2019-09-10'].copy(); BFUND = pd.Series(0.0001, index=BN.index)
else:
    OUT = ROOT / 'notebooks' / 'Turtle_Breakout' / '20'
    root = Path(os.environ['DERIBIT_DATA_ROOT']).parent
    YF = Path(os.environ.get('TB17_DATA', str(root / 'Stocks' / 'tb17' / 'btc_daily.parquet')))
    if not YF.exists():
        sys.path.insert(0, str(ROOT / 'tools')); import download_tb17_btc as dl; YF = dl.download(YF.parent)
    raw = pd.read_parquet(YF); raw['date'] = pd.to_datetime(raw['date']).dt.tz_localize(None).dt.normalize()
    YH = raw.set_index('date')[['open', 'high', 'low', 'close']].sort_index(); YH = YH[~YH.index.duplicated()].dropna(); YH = YH[(YH > 0).all(axis=1)]
    base = Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else root / 'Binance'
    BDIR = Path(os.environ.get('TB01_DATA_DIR', str(base / 'long')))
    BN = rv.daily_from_1m(pd.read_parquet(BDIR / 'BTCUSDT_um_1m.parquet', columns=['open_time', 'open', 'high', 'low', 'close']))
    fp = BDIR / 'basis' / 'funding' / 'BTCUSDT_funding.parquet'
    BFUND = rv.daily_funding(tb.funding_events(pd.read_parquet(fp)) if fp.exists() else None, BN.index)
OUT.mkdir(parents=True, exist_ok=True)
env = os.environ.copy(); env['PYTHONPATH'] = str(ROOT / 'src') + os.pathsep + env.get('PYTHONPATH', '')
chk = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_turtle_flat_legs.py'], cwd=str(ROOT), env=env, capture_output=True, text=True)
print(chk.stderr.strip().splitlines()[-1]); assert chk.returncode == 0
print('Yahoo', YH.index[0].date(), '->', YH.index[-1].date(), '| Binance daily', BN.index[0].date(), '->', BN.index[-1].date())
print(json.dumps(fl.registration(LP), indent=1))

## 1. Turtle state, signals, legs and the random-entry pools

In [ ]:
def build(daily, funding):
    W = bh.run_window(daily, str(daily.index[0].date()), None, P, RPt, ta.AParams(), 0, np.random.default_rng(0))
    TRl = W['trades'].query("kind == 'long'").copy(); TRl.loc[TRl.open_at_end, 'exit'] = daily.index[-1] + pd.Timedelta(days=1)
    state = fd.turtle_state(daily.index, TRl, W['long'].index[0]); flat = (state == 'flat'); r = np.log(daily.close).diff()
    s = {'daily': daily, 'funding': funding, 'flat': flat, 'r': r, 'turtle': W['long']}
    s['sig'] = {'A': rv.signals(r, flat, RV), 'B': fl.signals_b(daily, flat, LP)}
    s['side'] = {'A': 1, 'B': -1}
    s['T'], s['E'], s['pool'] = {}, {}, {}
    for leg in ('A', 'B'):
        s['T'][leg], s['E'][leg] = fl.simulate_leg(daily, s['sig'][leg], s['side'][leg], LP, funding)
        s['pool'][leg] = fl.pool(daily, flat & r.notna(), s['side'][leg], LP, funding)
    return s
S = {'yahoo': build(YH, None), 'binance': build(BN, BFUND)}
for k, s in S.items():
    print(k, {leg: (int(s['sig'][leg].sum()), len(s['T'][leg])) for leg in ('A', 'B')}, '(signals, trades)', '| FLAT days', int(s['flat'].sum()))

## 2. Decision

In [ ]:
def cut(T, a, b):
    return T[(T.entry >= pd.Timestamp(a)) & (T.entry <= pd.Timestamp(b))] if len(T) else T
WIN = {'yahoo 2014-19': ('yahoo', '2014-01-01', '2019-12-31'), 'yahoo 2020-': ('yahoo', '2020-01-01', '2100-01-01'), 'binance': ('binance', '2019-01-01', '2100-01-01')}
DEC = {}
for leg in ('A', 'B'):
    DEC[leg] = {}
    for j, (w, (src, a, b)) in enumerate(WIN.items()):
        DEC[leg][w] = fl.gate(cut(S[src]['T'][leg], a, b), cut(S[src]['pool'][leg], a, b), LP, LP.seed + 10 * j + (leg == 'B'))
    DEC[leg]['verdict'] = 'EDGE' if all(DEC[leg][w]['pass'] for w in WIN) else 'NO_EDGE'
print(json.dumps(DEC, indent=1, default=float))

## 3. Reported: trades, exits, regimes, years

In [ ]:
for src in ('yahoo', 'binance'):
    for leg in ('A', 'B'):
        T = S[src]['T'][leg]
        if not len(T):
            print(src, leg, 'no trades'); continue
        T = T.assign(regime=[bh.regime_of(t) for t in T.entry], year=T.entry.dt.year)
        print(f"{src} leg {leg}: {len(T)} trades, hold days median {T.hold_days.median():.0f} (p90 {T.hold_days.quantile(.9):.0f}), exits {T.reason.value_counts().to_dict()}, "
              f"top-5 share of R {T.R.nlargest(5).sum() / T.R.sum() if T.R.sum() else np.nan:.2f}")
        display(T.groupby('regime' if src == 'yahoo' else 'year').agg(trades=('R', 'size'), mean_R=('R', 'mean'), sum_R=('R', 'sum'), win=('ret', lambda x: (x > 0).mean()),
                                                                     mean_ret_pct=('ret', lambda x: 100 * x.mean()), hold=('hold_days', 'median')).round(3))

## 4. Accounts: turtle, legs and combinations (equal capital per sleeve)

In [ ]:
ACC = {}
for src in ('yahoo', 'binance'):
    s = S[src]; tur = s['turtle']; idx = tur.index
    curves = {'turtle': tur / tur.iloc[0]}
    for leg in ('A', 'B'):
        e = s['E'][leg].reindex(idx).ffill(); curves[leg] = e / e.iloc[0]
    curves['turtle+A'] = (curves['turtle'] + curves['A']) / 2; curves['turtle+B'] = (curves['turtle'] + curves['B']) / 2
    curves['turtle+A+B'] = (curves['turtle'] + curves['A'] + curves['B']) / 3
    h = s['daily'].close.reindex(idx); curves['hold'] = h / h.iloc[0]
    ACC[src] = curves
    m = pd.DataFrame({k: tk.metrics(v * 1e5, 1e5) for k, v in curves.items()}).T
    lr = pd.DataFrame({k: np.log(curves[k]).diff() for k in ('turtle', 'A', 'B')})
    print(src); display(m[['cagr', 'max_dd', 'calmar', 'sharpe', 'vol']].round(3)); print('daily correlations'); display(lr.corr().round(3))
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, src in zip(axes, ('yahoo', 'binance')):
    for k, col in (('hold', '0.6'), ('turtle', '#1f5aa6'), ('A', '#3a7d44'), ('B', '#c2571a'), ('turtle+A+B', '#b33f40')):
        ax.plot(ACC[src][k].index, ACC[src][k], color=col, label=k, lw=1.1)
    ax.set_yscale('log'); ax.legend(frameon=False); ax.grid(alpha=0.25); ax.set_title(src, loc='left', fontsize=10)
plt.tight_layout(); plt.show()

## 5. Save

In [ ]:
for src in ('yahoo', 'binance'):
    for leg in ('A', 'B'):
        S[src]['T'][leg].to_csv(OUT / f'20_trades_{src}_{leg}.csv', index=False)
    pd.DataFrame(ACC[src]).to_csv(OUT / f'20_accounts_{src}.csv')
(OUT / '20_decision.json').write_text(json.dumps({'decision': DEC, 'registration': fl.registration(LP), 'synthetic': SYNTHETIC}, indent=1, default=float))
git = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], cwd=str(ROOT), capture_output=True, text=True).stdout.strip()
(OUT / '20_manifest.json').write_text(json.dumps({'version': fl.VERSION, 'git': git, 'python': platform.python_version(), 'created_utc': pd.Timestamp.now('UTC').isoformat(), 'synthetic': SYNTHETIC}, indent=1))
print('saved to', OUT); print({leg: DEC[leg]['verdict'] for leg in DEC})